In [ ]:
import json
import os
import pandas as pd
import numpy as np
import nltk
#nltk.download('punkt')
#nltk.download('stopwords')
import re
nltk.corpus.stopwords.words('english')
from nltk.corpus import stopwords

from wordcloud import WordCloud
from matplotlib import pyplot as plt

import re
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

### Analysis of Annotations

In [ ]:
survey_data = pd.read_csv("../annotations/participants/3 Step (Order Rank) - Given an Excel sheet with columns for Red and NIR reflectance values, write Python code to compute NDVI for each row and assign a land cover category (‘vegetation’, ‘urban’, or ‘transition’) based on NDVI thresholds..csv", skiprows=1)
question = "Given an Excel sheet with columns for Red and NIR reflectance values, write Python code to compute NDVI for each row and assign a land cover category (‘vegetation’, ‘urban’, or ‘transition’) based on NDVI thresholds."
survey_data_1 = survey_data.reset_index()

def plot_results(input_survey_data, question):
    input_survey_data['Display Name'] = input_survey_data['Display Name'].str.replace('Participant ', 'Annotator_')
    input_survey_data['Display Name'] = input_survey_data['Display Name'].str.replace(r'Annotator_(\d+)', lambda m: f'Annotator_{int(m.group(1)) - 1}', regex=True)

    # Define the mapping for options to reasoning types
    option_mapping = {
        'option_1': 'Reasoning with Doc',
        'option_2': 'Reasoning with Sections',
        'option_3': 'Manual Reasoning',
        'option_4': 'No Reasoning'
    }

    # Replace Opinion column values using the mapping
    input_survey_data['Opinion'] = input_survey_data['Opinion'].map(option_mapping)

    input_survey_data.rename(columns={'Opinion': 'Reasoning Approach'}, inplace=True)
    # Pivot the dataframe to get annotators as rows and reasoning types as columns
    output_data = input_survey_data.pivot(index='Display Name', columns='Reasoning Approach', values='Rating')

    # Sort columns in the desired order
    column_order = ['No Reasoning', 'Reasoning with Doc', 'Reasoning with Sections', 'Manual Reasoning']
    output_data = output_data[column_order]
    plt.figure(figsize=(20, 10))
    sns.heatmap(output_data, cmap='RdYlGn', cbar=True, annot=True, fmt='g', annot_kws={'fontsize': 15})
    #plt.title(f'Rater Scores per Code Version \n {question}', fontsize=14)
    plt.xlabel('Reasoning Approach', fontsize=15)
    plt.ylabel('Annotator', fontsize=15)
    plt.xticks(fontsize=15)
    plt.yticks(fontsize=15)
    plt.show()
    plt.close()

    plt.figure(figsize=(20, 10))
    summary_stats = input_survey_data.groupby('Reasoning Approach')['Rating'].agg(['median', 'std']).reindex(column_order)

    # fill NaN std (single sample) and drop approaches with no median
    summary_stats['std'] = summary_stats['std'].fillna(0)
    summary_stats = summary_stats.dropna(subset=['median'])

    # Create bar plot with error bars
    x_coords = np.arange(len(summary_stats))
    ax = sns.barplot(x=summary_stats.index, y=summary_stats['median'].values, errorbar=None, palette='RdYlGn')
    ax.errorbar(x_coords, summary_stats['median'].values, yerr=summary_stats['std'].values,
                fmt='none', ecolor='k', capsize=6, linewidth=1.5)

    #plt.title(f'Average Rater Scores with Std Dev \\n {question}', fontsize=14)
    plt.xlabel('Reasoning Approach', fontsize=20)
    plt.ylabel('Median Rating', fontsize=20)
    plt.xticks(fontsize=20)
    plt.yticks(fontsize=20)
    plt.tight_layout()
    plt.show()

plot_results(survey_data_1.copy(), question)

In [ ]:
survey_data_3 = pd.read_csv("../annotations/participants/5 Step (Order Rank) - Given the spectral band values for a pixel [Red = 0.15, Green = 0.35, Blue = 0.20, NIR = 0.60], write Python code to compute NDVI and classify the pixel as ‘vegetation’, ‘water’, or ‘urban’ based on NDVI thresholds..csv", skiprows=1)
question = "Given the spectral band values for a pixel: [Red = 0.15, Green = 0.35, Blue = 0.20, NIR = 0.60], write Python code to compute NDVI and classify the pixel as ‘vegetation’, ‘water’, or ‘urban’ based on NDVI thresholds."
plot_results(survey_data_3.copy(), question)

In [ ]:
survey_data_4 = pd.read_csv("../annotations/participants/6 Step (Order Rank) - Given the red and near-infrared (NIR) reflectance values of a pixel as floating-point numbers, write Python code to compute the NDVI (NIR - Red)  (NIR + Red). Return 0 if the denominator is zero, and classify the pixel as.csv", skiprows=1)
question = "Given the red and near-infrared (NIR) reflectance values of a pixel as floating-point numbers, write Python code to compute the NDVI: (NIR - Red) / (NIR + Red). Return 0 if the denominator is zero, and classify the pixel as ‘vegetation’ if NDVI > 0.3, ‘urban’ if NDVI < 0.2, and ‘transition’ otherwise."
plot_results(survey_data_4.copy(), question)

In [ ]:
survey_data_6 = pd.read_csv("../../annotations/participants/8 Step (Order Rank) - Write a Python function that takes the spectral band values of a pixel (at minimum Red and NIR) as input, computes NDVI, and classifies the land cover as ‘vegetation’, ‘water’, or ‘urban’ using defined thresholds.csv", skiprows=1)
question = "Write a Python function that takes the spectral band values of a pixel (at minimum Red and NIR) as input, computes NDVI, and classifies the land cover as ‘vegetation’, ‘water’, or ‘urban’ using defined thresholds."
plot_results(survey_data_6, question)

##### Domain Questions

In [ ]:
survey_data_2 = pd.read_csv("../annotations/participants/4 Step (Order Rank) - Write Python code to compute NDVI from a GeoTIFF file with Red and NIR bands, and classify each pixel into ‘vegetation’, ‘urban’, or ‘transition’ using thresholds of 0.3 and 0.2. Save the output as a new classified raster.csv", skiprows=1)
question = "Write Python code to compute NDVI from a GeoTIFF file with Red and NIR bands, and classify each pixel into ‘vegetation’, ‘urban’, or ‘transition’ using thresholds of 0.3 and 0.2. Save the output as a new classified raster file."
plot_results(survey_data_2.copy(), question)

In [ ]:
survey_data_5 = pd.read_csv("../annotations/participants/7 Step (Order Rank) - Given a Sentinel-2 GeoTIFF image containing Red and NIR bands, write Python code to compute NDVI and classify the land cover into ‘vegetation’, ‘urban’, or ‘water’..csv", skiprows=1)
question = "Given a Sentinel-2 GeoTIFF image containing Red and NIR bands, write Python code to compute NDVI and classify the land cover into ‘vegetation’, ‘urban’, or ‘water’."
plot_results(survey_data_5.copy(), question)

In [ ]:
survey_data_7 = pd.read_csv("../annotations/participants/9 Step (Order Rank) - Given a Sentinel-2 GeoTIFF file, write a Python code to compute NDVI for every pixel and classify each pixel as ‘urban’ if NDVI  0.2, ‘vegetation’ if NDVI  0.3, and ‘transition’ otherwise..csv", skiprows=1)
question = "Given a Sentinel-2 GeoTIFF file, write  a Python code to compute NDVI for every pixel and classify each pixel as ‘urban’ if NDVI < 0.2, ‘vegetation’ if NDVI > 0.3, and ‘transition’ otherwise."
plot_results(survey_data_7.copy(), question)

In [ ]:
def combine_results(input_survey_data_ls):
    combined_df_ls = []
    for idx, input_survey_data in enumerate(input_survey_data_ls):
        input_survey_data['Display Name'] = input_survey_data['Display Name'].str.replace('Participant', 'Annotator_')

        # Define the mapping for options to reasoning types
        option_mapping = {
            'option_1': 'Reasoning with Doc',
            'option_2': 'Reasoning with Sections',
            'option_3': 'Manual Reasoning',
            'option_4': 'No Reasoning'
        }

        # Replace Opinion column values using the mapping
        input_survey_data['Opinion'] = input_survey_data['Opinion'].map(option_mapping)

        input_survey_data.rename(columns={'Opinion': 'Reasoning Approach'}, inplace=True)
        # Pivot the dataframe to get annotators as rows and reasoning types as columns
        output_data = input_survey_data.pivot(index='Display Name', columns='Reasoning Approach', values='Rating')

        # Sort columns in the desired order
        column_order = ['No Reasoning', 'Reasoning with Doc', 'Reasoning with Sections', 'Manual Reasoning']
        output_data = output_data[column_order]
        combined_df_ls.extend([output_data])
    combined_df = pd.concat(combined_df_ls)
    return combined_df

combined_survey_df = combine_results([survey_data_1.copy(), survey_data_2.copy(), survey_data_3.copy(), survey_data_4.copy(), survey_data_5.copy(), survey_data_6.copy(), survey_data_7.copy()])
combined_survey_df = combined_survey_df.reset_index()


##### Annotator preferences

In [ ]:
all_surveys = pd.concat([survey_data_1.copy(), survey_data_2.copy(), survey_data_3.copy(), survey_data_4.copy(), survey_data_5.copy(), survey_data_6.copy(), survey_data_7.copy()])
all_surveys = all_surveys.reset_index()
all_surveys['Display Name'] = all_surveys['Display Name'].str.replace('Participant ', 'Annotator_')
all_surveys['Display Name'] = all_surveys['Display Name'].str.replace(r'Annotator_(\d+)', lambda m: f'Annotator_{int(m.group(1)) - 1}', regex=True)
all_surveys.head()

In [ ]:
all_surveys_copy = all_surveys.copy()

# Define the mapping for options to reasoning types
option_mapping = {
    'option_1': 'Doc Level Reasoning',
    'option_2': 'Section Level Reasoning',
    'option_3': 'Manual Reasoning',
    'option_4': 'No Reasoning'
}

# Replace Opinion column values using the mapping
all_surveys_copy['Opinion'] = all_surveys_copy['Opinion'].map(option_mapping)

# Group by Display Name and Opinion, then calculate median
annotator_summary = all_surveys_copy.groupby(['Display Name', 'Opinion'])['Rating'].median().unstack()

# Sort columns in the desired order
column_order = ['No Reasoning', 'Doc Level Reasoning', 'Section Level Reasoning', 'Manual Reasoning']
annotator_summary = annotator_summary[column_order]

# Display the summary table
print("Median Ratings per Annotator by Reasoning Approach:")
print(annotator_summary)

# Create a heatmap visualization
plt.figure(figsize=(12, 8))
sns.heatmap(annotator_summary, 
            annot=True, 
            fmt='.1f', 
            cmap='RdYlGn', 
            #cbar_kws={'label': ''},
            vmin=annotator_summary.min().min(), 
            vmax=annotator_summary.max().max(),
            annot_kws={'fontsize': 12},
            linewidths=0.5,
            linecolor='gray')

#plt.title("Median Ratings per Annotator by Reasoning Approach", fontsize=16, pad=20)
plt.xlabel("Reasoning Approach", fontsize=14)
plt.ylabel("Annotator", fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_annotator_radar(annotator):
    option_mapping = {
    'option_4': 'Direct Generation',
    'option_1': 'Unified Reasoning',
    'option_2': 'Modular Reasoning',
    'option_3': 'Expert-Crafted Reasoning'
    }

    annotator['Opinion'] = annotator['Opinion'].map(option_mapping)

    # Define custom order (Direct Generation at top)
    column_order = ['Direct Generation', 'Modular Reasoning', 'Expert-Crafted Reasoning', 'Unified Reasoning']
    means = annotator.groupby('Opinion')['Rating'].mean().reindex(column_order)

    # Radar setup
    categories = means.index
    values = means.values
    N = len(categories)

    # Angles spaced around the circle
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    values = np.concatenate((values, [values[0]]))
    angles += angles[:1]

    plt.figure(figsize=(16, 16))
    ax = plt.subplot(111, polar=True)

    # Rotate so that the first axis (Direct Generation) is at the top
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    # Plot and fill
    ax.plot(angles, values, color='red', linewidth=2)
    ax.fill(angles, values, color='teal', alpha=0.25)

    # Remove radial labels
    ax.set_yticklabels([])

    # Set up the category labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=35)

    # Adjust label alignment for side categories
    for label, angle in zip(ax.get_xticklabels(), angles[:-1]):
        angle_deg = np.degrees(angle)
        if 70 < angle_deg < 110:  # roughly top-right (Unified)
            label.set_horizontalalignment('left')
            label.set_rotation(45)
        elif 250 < angle_deg < 290:  # bottom-left (Modular)
            label.set_horizontalalignment('right')
            label.set_rotation(-45)
        elif 160 < angle_deg < 200:  # bottom (Expert-Crafted)
            label.set_verticalalignment('top')
        elif 340 < angle_deg or angle_deg < 20:  # top (Direct Generation)
            label.set_verticalalignment('bottom')

    # Optionally push labels outward for clarity
    ax.tick_params(pad=15)
    plt.tight_layout()
    plt.show()


In [ ]:
annotator_1 = all_surveys[all_surveys['Display Name'] == 'Evaluator_1']
plot_annotator_radar(annotator_1)